# IMPORT API KEY

In [1]:
import getpass
import os

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")

# SEARCH TOOL DEFINITION

In [2]:
from typing import List, Literal, Optional
from langchain.tools import tool
from langchain_tavily import TavilySearch

@tool
def tavily_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_answer: bool = False,
    include_raw_content: bool = False,
    include_images: bool = False,
    include_image_descriptions: bool = False,
    search_depth: Literal["basic", "advanced"] = "basic",
    time_range: str = "",            # instead of None
    include_domains: List[str] = [], # instead of None
    exclude_domains: List[str] = [], # instead of None
    country: str = ""                # instead of None
) -> str:
    """
    Perform a web search using TavilySearch.

    Args:
        query: The search query string.
        max_results: Maximum number of search results to return.
        topic: Category of the search ("general", "news", "finance").
        include_answer: Include an answer to the original query in results.
        include_raw_content: Include cleaned and parsed HTML of each search result.
        include_images: Include a list of query-related images in the response.
        include_image_descriptions: Include descriptive text for each image.
        search_depth: Depth of the search ("basic" or "advanced").
        time_range: Time range to filter results ("day", "week", "month", "year") or "".
        include_domains: List of domains to specifically include.
        exclude_domains: List of domains to specifically exclude.
        country: Country to focus the search on.

    Returns:
        Search results as a string.
    """
    # parameters validation
    if len(time_range) == 0:
        time_range = None
    if len(include_domains) == 0:
        include_domains = None
    if len(exclude_domains) == 0:
        exclude_domains = None
    if len(country) == 0:
        country = None

    tool_instance = TavilySearch(
        max_results=max_results,
        topic=topic,
        include_answer=include_answer,
        include_raw_content=include_raw_content,
        include_images=include_images,
        include_image_descriptions=include_image_descriptions,
        search_depth=search_depth,
        time_range=time_range,
        include_domains=include_domains,
        exclude_domains=exclude_domains,
        country=country
    )

    return tool_instance.invoke({"query": query})

# EXTRACT TOOL DEFINITION

In [3]:
from typing import Literal
from langchain.tools import tool
from langchain_tavily import TavilyExtract

@tool
def tavily_extract(
    url: list,
    extract_depth: Literal["basic", "advanced"] = "basic",
    chunks_per_source: Literal[1, 2, 3, 4, 5] = 3,
    include_images: bool = False,
    include_favicon: bool = False,
    format_text: Literal["markdown", "text"] = "markdown"
) -> str:
    """
    Extract content from a web page using TavilyExtract.

    Args:
        url: The URL of the pages to extract content from.
        extract_depth: Extraction depth ("basic" or "advanced").
        chunks_per_source: Number of chunks per source (1-5).
        include_images: Whether to include images (True or False).
        include_favicon: Whether to include favicon (True or False).
        format_text: Output format ("markdown" or "text").

    Returns:
        Extracted content as a string.
    """

    tool_instance = TavilyExtract(
        extract_depth=extract_depth,
        chunks_per_source=chunks_per_source,
        include_images=include_images,
        include_favicon=include_favicon,
        format=format_text,
    )

    return tool_instance.invoke({"urls": url})

# AGENT DEFINITION

In [8]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:9b")

agent = create_agent(
    model=model,
    tools=[tavily_search, tavily_extract],
    checkpointer=InMemorySaver(),
    # system_prompt = """
    #     You are a web research assistant. You can execute the 'tavily_search' tool to perform a web search, and the 
    #     'tavily_extract' tool to perform the extraction of the webpage.

    #     The 'tavily_search' tool must be executed according to these rules:
    #     - The user's input will be in two parts: 'text' (query) and 'params' (dictionary of tool parameters).
    #     - The user will always provide a search query and a dictionary of parameters for the tool.
    #     - Only use the parameters provided by the user.
    #     - Execute the tool using these parameters.

    #     Make sure to use the 'tavily_extract' tool with the following parameters:
    #     - extract_depth = "basic"
    #     - chunks_per_source = 3
    #     - include_images = False
    #     - include_favicon = False
    #     - format_text = "markdown"
    #     Retrieve the entire original content from the user defined webpage. 
    #     Do NOT modify it at all.
    #     """,
    # system_prompt = """
    #     You are a web research assistant. You can use the 'tavily_search' tool to perform web searches, and the 'tavily_extract' 
    #     tool to extract webpage content. 
    #     The user will ask or for a web search or for a webpage extraction. If the user provides a query with some parameters, use 
    #     the 'tavily_search' tool. If the user provides a list of url, use the 'tavily_extract' tool. 
    
    #     Rules for using the 'tavily_search' tool:
    #     The user's input will consist of two parts: 'text' (the query) and 'params' (a dictionary of tool parameters).
    #     The user will always provide a search query and a set of parameters for the tool.
    #     Only use the parameters provided by the user.
    #     Execute the tool using exactly the parameters given.
        
    #     Guidelines for using the 'tavily_extract' tool:
    #     Retrieve the entire original content from the user-defined webpage. The extraction must be an exact replica of the tool 
    #     results without any modifications (no changes to the lexicon, punctuation, format, or language). The output must be 
    #     identical to the tool's result, with no adjustments or alterations whatsoever.
    #     Please strictly follow the rules below:
    #     Set extract_depth to "basic".
    #     Set chunks_per_source to 3.
    #     Set include_images to False.
    #     Set include_favicon to False.
    #     Set format_text to "markdown".
    #     """,
    system_prompt = """
    You are a web research assistant. You can use the 'tavily_search' tool to perform web searches, and the 'tavily_extract' 
    tool to extract webpage content. 
    The user will ask or for a web search or for a webpage extraction. If the user provides a query with some parameters, use 
    the 'tavily_search' tool. If the user provides a list of url, use the 'tavily_extract' tool. 
    Rules for using the 'tavily_search' tool:
    The user's input will consist of two parts: 'text' (the query) and 'params' (a dictionary of tool parameters).
    The user will always provide a search query and a set of parameters for the tool.
    Only use the parameters provided by the user.
    Execute the tool using exactly the parameters given.
    
    Guidelines for using the 'tavily_extract' tool:
    Retrieve the entire original content from the user-defined webpage. The extraction must be an exact replica of the tool 
    results without any modifications (no changes to the lexicon, punctuation, format, or language). The output must be 
    identical to the tool's result, with no adjustments or alterations whatsoever.
    
    REPORTING RULES (CRITICAL) ONLY for the 'tavily_extract' tool:
    - Your final response MUST be a verbatim report of the extracted content.
    - DO NOT modify the content at all (no changes to lexicon, punctuation, format, or language).
    - DO NOT summarize the content.
    - DO NOT add any introductory or concluding text (e.g., "Here are the extracted recipes").
    - DO NOT add any AI-generated headers or markdown structures not present in the original tool output.
    - Your output must consist ONLY of the SOURCE headers and the verbatim content from the tool.
    - For EACH source in the tool result, format it EXACTLY like this:
        
        ----- SOURCE: <url>
        <raw_content>
        
    - Separate different sources with TWO NEWLINES. Example of two results retrieved from the <tavily_extract> tool:

        ----- SOURCE: <url_1>
        <raw_content_1>


        ----- SOURCE: <url_2>
        <raw_content_2>
    
    Please strictly follow the tool parameters below:
    Set extract_depth to "basic".
    Set chunks_per_source to 3.
    Set include_images to False.
    Set include_favicon to False.
    Set format_text to "markdown".
    """,
)

# WEB SEARCH INVOCATION

In [9]:
import json

default_params = {    
    'max_results': 5,
    'topic': "general",
    'include_answer': False,
    'include_raw_content': False,
    'include_images': False,
    'include_image_descriptions': False,
    'search_depth': "basic",
    'time_range': "",
    'include_domains': [],
    'exclude_domains': [],
    'country': "",
}
user_params = {"include_domains": ["*.it"], "exclude_domains": ["*.com", "wikipedia.org"], "country": "italy"}
params = {**default_params, **user_params}
params_json = json.dumps(params)

user_query = "\"Pasta alla carbonara\""
query = f"text: {user_query}, params: {params_json}"

result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
    {"configurable": {"thread_id": "1"}},
)

###### NOT STRUCTURED RESULT

In [10]:
result

{'messages': [HumanMessage(content='text: "Pasta alla carbonara", params: {"max_results": 5, "topic": "general", "include_answer": false, "include_raw_content": false, "include_images": false, "include_image_descriptions": false, "search_depth": "basic", "time_range": "", "include_domains": ["*.it"], "exclude_domains": ["*.com", "wikipedia.org"], "country": "italy"}', additional_kwargs={}, response_metadata={}, id='1586790f-a628-4dce-9eb4-2af4f0b39880'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5:9b', 'created_at': '2026-04-08T14:28:08.5902243Z', 'done': True, 'done_reason': 'stop', 'total_duration': 382159490900, 'load_duration': 11290158600, 'prompt_eval_count': 1361, 'prompt_eval_duration': 60017738300, 'eval_count': 235, 'eval_duration': 309922197600, 'model_name': 'qwen3.5:9b', 'model_provider': 'ollama'}, id='lc_run--019d6d78-7e39-7560-9c21-755e8ce5c566-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'Pasta alla carbonara', 'inc

### Results filtering

In [11]:
overview = result["messages"][-1]
overview.pretty_print()

================================== Ai Message ==================================

Here are the search results for "Pasta alla carbonara":

---

**1. Tuduu - Pasta alla carbonara**
🔗 https://tuduu.it/en/recipes/tuduu/pasta-alla-carbonara

Pasta alla carbonara is a classic dish of Roman cuisine. It is a simple but irresistible first course, made with few ingredients but with an intense flavor. The creamy sauce is made with eggs, pancetta, Pecorino Romano and black pepper.

**Preparation time:** 30 min

**Ingredients:**
- Sweet Guanciale 250g - Mendelspeck
- Zabaglione 220g
- Roman sheep cheese dop about 250 g - bronzetto
- Garlic oil chili packet 20g

---

**2. Italia Regina - Carbonara Spaghetti**
🔗 https://www.italiaregina.it/carbonara-spaghetti/

Carbonara Spaghetti is an Italian pasta dish from Rome based on eggs, cheese (Pecorino Romano or Parmigiano-Reggiano), bacon (guanciale or pancetta), and black pepper.

**Common pasta shapes:**
- Spaghetti (traditional)
- Fettuccine
- Rigaton

In [13]:
import json

tool_msg = json.loads(result["messages"][-2].content)

research_results = tool_msg["results"]
research_results

[{'url': 'https://tuduu.it/en/recipes/tuduu/pasta-alla-carbonara',
  'title': 'Pasta alla carbonara',
  'content': "# Pasta alla carbonara. Pasta alla carbonara is a classic dish of Roman cuisine. It is a simple but irresistible first course, made with few ingredients but with an intense flavor. The creamy sauce is made with eggs, pancetta, Pecorino Romano and black pepper. Preparation time: 30 minPreparation: 30 min. ## Ingredients. ### Sweet Guanciale 250g - Mendelspeck. ### Zabaglione 220g. * Roman sheep cheese dop about 250 g - bronzetto. ### Roman sheep cheese dop about 250 g - bronzetto. * Garlic oil chili packet 20g. ### Garlic oil chili packet 20g. ## Preparation. Drain the pasta directly into the resulting creamy sauce and mix off the heat. If necessary, add a ladle of the pasta cooking water for a creamier result. | of which Sugars (g) | 2.2 |. | of which Saturates (g) | 1.81 |. The data represented here, limited to certain specificities, are the result of an analysis carried

# WEB PAGE EXTRACTION INVOCATION

In [14]:
from langchain.messages import HumanMessage

query = HumanMessage(content=[
    {"type": "text", "text": [research_results[0]["url"], research_results[-1]["url"]]},
])

response = agent.invoke(
    {"messages": [query]},
    {"configurable": {"thread_id": "1"}},
)

In [15]:
# DISPLAY TOOL MESSAGE
from IPython.display import Markdown, display

display(Markdown(response["messages"][-2].content))

{"results": [{"url": "https://tuduu.it/en/recipes/tuduu/pasta-alla-carbonara", "title": "Pasta alla carbonara", "raw_content": "[How it works](/en/about/how-tuduu-works-the-digital-marketplace-of-made-in-italy)\n\nFor consumersPersonal purchases\n\n![Pasta alla carbonara](https://tuduu-prd-assets-fde-ghdcd5e6baagctam.z01.azurefd.net/recipesimages/recipe-17278.jpg?v=2)\n\n# Pasta alla carbonara\n\n@tuduu\n\n[Category: First courses](/en/recipes/categories/first-courses)\n\nPasta alla carbonara is a classic dish of Roman cuisine. It is a simple but irresistible first course, made with few ingredients but with an intense flavor. The creamy sauce is made with eggs, pancetta, Pecorino Romano and black pepper. Carbonara is a true comfort food, perfect to enjoy during cold winter evenings.\n\nDifficulty: Easy\n\nCooking time: minCooking: min\n\nPreparation time: 30 minPreparation: 30 min\n\nCountry: Italia\n\ntuduu@tuduu\n\n## Ingredients\n\nNo. Servings\n\n* Spaghetti350350\n* ![Guanciale](https://static.spaghettiemandolino.it/cdn-cgi/image/format=auto,fit=contain,width=500,height=500/img_prodotti/big/5731-guanciale-dolce-mendelspeck.png?1729593433)\n\n  Guanciale120120\n* ![Egg yolks](https://static.spaghettiemandolino.it/cdn-cgi/image/format=auto,fit=contain,width=500,height=500/img_prodotti/big/4912-zabaione-220g.png?1692365817)\n\n  Egg yolksq.b.\n* ![Pecorino Romano cheese](https://cdn.shopify.com/s/files/1/0025/0141/0873/products/Costanzo_Dispensa_0091_IMG_3041_risultato_jpg.jpg?v=1591617223)\n\n  Pecorino romano cheese44\n* ![Pepper](https://static.spaghettiemandolino.it/cdn-cgi/image/format=auto,fit=contain,width=500,height=500/img_prodotti/big/5004-aglio-olio-peperoncino-bustina-da-20g.png?1750088685)\n\n  Pepperq.b.\n* Saltq.b.\n\n## Purchasable products\n\n* ![Sweet Guanciale 250g - Mendelspeck](https://static.spaghettiemandolino.it/cdn-cgi/image/format=auto,fit=contain,width=500,height=500/img_prodotti/big/5731-guanciale-dolce-mendelspeck.png?1729593433)\n\n  ### Sweet Guanciale 250g - Mendelspeck\n\n  1 product1 product\n\n  £ 5.48\n\n* ![Zabaglione 220g](https://static.spaghettiemandolino.it/cdn-cgi/image/format=auto,fit=contain,width=500,height=500/img_prodotti/big/4912-zabaione-220g.png?1692365817)\n\n  ### Zabaglione 220g\n\n  1 product1 product\n\n  £ 7.31\n\n* ![Roman sheep cheese dop about 250 g - bronzetto](https://cdn.shopify.com/s/files/1/0025/0141/0873/products/Costanzo_Dispensa_0091_IMG_3041_risultato_jpg.jpg?v=1591617223)\n\n  ### Roman sheep cheese dop about 250 g - bronzetto\n\n  1 product1 product\n\n  £ 7.40\n\n* ![Garlic oil chili packet 20g](https://static.spaghettiemandolino.it/cdn-cgi/image/format=auto,fit=contain,width=500,height=500/img_prodotti/big/5004-aglio-olio-peperoncino-bustina-da-20g.png?1750088685)\n\n  ### Garlic oil chili packet 20g\n\n  1 product1 product\n\n  £ 1.04\n\n* ![Gray non-stick pan 1965 Vintage Quarzo Nero](https://static.spaghettiemandolino.it/cdn-cgi/image/format=auto,fit=contain,width=500,height=500/img_prodotti/big/2639-padella-antiaderente-grigia-1965-vintage-quarzo-nero.jpeg?1692363115)\n\n  ### Gray non-stick pan 1965 Vintage Quarzo Nero\n\n  £ 43.33\n\n## Preparation\n\n1. STEP 1 OF 3\n\n   Dice the guanciale and brown it in a pan until it becomes crispy. Set aside some of its fat.\n2. STEP 2 OF 3\n\n   Beat the egg yolks with the Pecorino and pepper. Add the guanciale fat. Drain the pasta directly into the resulting creamy sauce and mix off the heat.\n3. STEP 3 OF 3\n\n   If necessary, add a ladle of the pasta cooking water for a creamier result. Serve with another grind of pepper and, if you like, more Pecorino Romano.\n\n## Suggestions\n\n* Pan\n* Bowl\n\n## General Information\n\n### Origin\n\nItalia, Lazio\n\n## Analysis\n\n## Macronutrients(100 gr)\n\n## FAQs\n\n|  |  |\n| --- | --- |\n| Energy (kcal) | 425.93 |\n| Carbohydrates (g) | 48.82 |\n| of which Sugars (g) | 2.2 |\n| Fat (g) | 19.87 |\n| of which Saturates (g) | 1.81 |\n| Protein (g) | 12.55 |\n| Fiber (g) | 1.82 |\n| Sale (g) | 0.19 |\n\n* Proteins\n\n  12.55g · 12%\n* Carbohydrates\n\n  48.82g · 46%\n* Fats\n\n  19.87g · 42%\n\n##### Attention\n\nThe data represented here, limited to certain specificities, are the result of an analysis carried out using Tuduu's proprietary algorithms. As such, they may contain errors and/or inaccuracies, therefore users are always requested to verify their correctness. If anomalies are detected, please contact us at[info@tuduu.it](mailto:info@tuduu.it)\n\nLifestyles\n\nIntolerances and allergies", "images": []}, {"url": "https://www.ceramicavietrese.it/en/recipes-claudia/a-typical-italian-food--pasta-alla-carbonara-a-typical-recipe-of-rome-tradition-and-italian-tradition/", "title": "Pasta alla \"Carbonara\" a typical recipe of Rome tradition ...", "raw_content": "[![VIETRI CERAMICS - the excellence artisan pottery made in Italy]()  VIETRI CERAMICS - the excellence artisan pottery made in Italy](/en/) \n\n[0](/en/cart/)  [Login](/en/login/)\n\n* [Italiano](/)\n* [English](/en/)\n* [Español](/es/)\n* [Deutsch](/de/)\n* [Français](/fr/)\n* [Pусский](/ru/)\n* [中国](/c2/)\n\n* [Home](/en/)\n* [About us](#) \n  + [Vietri and the ceramic](/en/vietri-and-the-ceramic/)\n  + [Our master potters](/en/our-master-potters/)\n  + [The art of manufacturing ceramic](/en/the-art-of-manufacturing-ceramic/)\n  + [Our design furniture](/en/our-design-furniture/)\n  + [Photo Gallery](/en/photo-gallery/)\n  + [Video-Gallery](/en/video-gallery/)\n  + [Shipments](/en/shipments/)\n  + [Payment methods](/en/payment-methods/)\n  + [Warranty, defects and refunds](/en/warranty-defects-and-refunds/)\n  + [Sure because we have MOCA certification](/en/sure-because-we-have-moca-certification/)\n  + [FAQ Frequently Asked Questions](/en/faq-frequently-asked-questions/)\n  + [Latest News](/en/news/)\n  + [Recipes Claudia](/en/recipes-claudia/)\n* [Products Products](#) \n  + [Complete catalog](/en/products/)\n\n    [Prompt Delivery](/en/products/?filter=stock)\n\n    [Complete catalog](/en/products/)  [Prompt Delivery](/en/products/?filter=stock)\n\n     [![Spice Jar with Royal House Decor 2]()](/en/products/spice-jar-with-royal-house-decor-2/?cat=ceramic-jars)\n\n    ### [On the table and in the kitchen](/en/products/on-the-table-and-in-the-kitchen/)\n\n    - [Dishes](/en/products/dishes/)\n    - [Chef and Gourmet Shapes Dishes](/en/products/chef-and-gourmet-shapes-dishes/)\n    - [PizzaPiatto®](/en/products/pizzapiatto/)\n    - [Breakfast](/en/products/breakfast/)\n    - [Lunch Dinner](/en/products/lunch-dinner/)\n    - [Bowls, Trays and Serving Plates](/en/products/bowls-trays-and-serving-plates/)\n    - [Center Table](/en/products/center-table/)\n    - [Accessories](/en/products/accessories/)\n    - [Ceramic jars](/en/products/ceramic-jars/)\n    - [Terrine da Forno](/en/products/terrine-da-forno/)\n    - [Trays in wood and tiles](/en/products/trays-in-wood-and-tiles/)\n\n    ### [Wedding favors](/en/products/wedding-favors/)\n\n    - [Wedding](/en/products/wedding/)\n    - [Baptism](/en/products/baptism/)\n    - [Birthday, Anniversary, Graduation](/en/products/birthday-anniversary-graduation/)\n    - [First Holy Communion](/en/products/first-holy-communion/)\n    - [Magnets](/en/products/magnets/)\n\n    ### [Home furnishing](/en/products/home-furnishing/)\n\n    - [Bathroom furniture](/en/products/bathroom-furniture/)\n    - [Vasi e Cachepot](/en/products/vasi-e-cachepot/)\n    - [Wall Plates](/en/products/wall-plates/)\n    - [Umbrella stand](/en/products/umbrella-stand/)\n    - [Lumi e Lampshade](/en/products/lumi-e-lampshade/)\n    - [Objects](/en/products/objects/)\n    - [Tiles, Tiles, Floors, Coverings and Kitchen Countertops](/en/products/tiles-tiles-floors-coverings-and-kitchen-countertops/)\n    - [Panels - Murals made of project](/en/products/panels--murals-made-of-project/)\n    - [Tables and Coffee Tables](/en/products/tables-and-coffee-tables/)\n\n    ### [Garden furniture](/en/products/garden-furniture/)\n\n    - [Kitchen Tables, Coffee Tables, Bench and Outdoor Fountain](/en/products/kitchen-tables-coffee-tables-bench-and-outdoor-fountain/)\n    - [Chairs and Stools](/en/products/chairs-and-stools/)\n    - [Fountains](/en/products/fountains/)\n\n    ### [Design furniture](/en/products/design-furniture/)\n\n    - [Bar tables and stools](/en/products/bar-tables-and-stools/)\n    - [Tables and Coffee Tables](/en/products/tables-and-coffee-tables1/)\n    - [Benches and Benches](/en/products/benches-and-benches/)\n\n    ### [Festivity](/en/products/festivity/)\n\n    - [Christmas](/en/products/christmas/)\n    - [Easter](/en/products/easter/)\n\n    ### [Artistic](/en/products/artistic/)\n\n    In this section you will find High Ceramics objects, UNIQUE PIECES created by our. Master ...\n\n    ### [Outlet - Discounts up to 60%](/en/products/outlet-discounts-up-to-60/)\n\n    Sales: Here you can find items DISCOUNTED up to 60% end of series or ex display or with small ...\n* [Gift Card](/en/gift-card/)\n* [Lists  Gifts](#) \n  + [Wedding list](/en/lists-online/wedding-list/)\n  + [Gifts list](/en/lists-online/gifts-list/)\n  + [Create a list](/en/login/?from=lc)\n  + [Search a wedding list](/en/search-for-a-gift-list/?cat=wedding-list)\n  + [Search for a gift list](/en/search-for-a-gift-list/?cat=gifts-list)\n  + [Give a Gift Card](/en/gift-card/)\n* [Contacts](/en/contacts/)\n\n* [Italiano](/)\n* [English](/en/)\n* [Español](/es/)\n* [Deutsch](/de/)\n* [Français](/fr/)\n* [Pусский](/ru/)\n* [中国](/c2/)\n\n[0](/en/cart/ \"Cart\")\n\n* [Home](/en/)\n* [Recipes Claudia](/en/recipes-claudia/)\n* A typical Italian Food - ...\n\n## Recipes Claudia\n\n[Back to the list](/en/recipes-claudia/)\n\n# A typical Italian Food - Pasta alla \"Carbonara\" a typical recipe of Rome tradition and Italian tradition\n\nA typical recipe of Italian traditions , especially Roman traditions . Easy to do, but that once tasted you can hardly forget . Tradition would like to use only seasoned and grated Pecorino Romano, but if you do not like the taste too strong , you can replace a portion of cheese with Original Parmigiano Reggiano (or original Grana Padano ) that definitely has a taste and a flavor less \"strong \" .\n\n[![image](/files/admin/immagini/pasta-alla-carbonara-ricetta-tipica-in-ceramica-di-vietri-ceramica-vietrese4-l-20200910230002.jpg)](/files/admin/immagini/pasta-alla-carbonara-ricetta-tipica-in-ceramica-di-vietri-ceramica-vietrese4-l-20200910230002.jpg \"Pork cheek\") \n\nPork cheek\n\nRecipe for 4 people\n\n***Ingredients:***\n\n- 350 gr. of spaghetti of Gragnano\n\n- 3 eggs + 1 egg yolk\n\n- 160gr of cheek lard\n\n- Roman pecorino\n\n- Roman parmisan\n\n- salt and pepper q.b.\n\n***Preparazione:***\n\nPut a pot with abundant water on the fire and when it is going to boile, you'll salt. Use less salt as usual because pasta have already a tasty condiment with cheek lard, parmesan and pecorino.\n\nWhile you cut the cheek lard in cubes or strips, put in a pot on fire until the fatty become transparent and lightly crounch, then put off from the fire and let it lightly cold.\n\nBreak the eggs in a bowl and mix them. Add the pecorino, parmesan and the pepper (as you like) and mix to obtain an homogeneus mixture. Add the cheek lard.\n\nPut the pasta to cook.\n\nWhen pasta will be ready, dry it and put in the bowl and mix with the mixture and amalgamate them.\n\n***Serve Spaghetti alla Carbonara immediately, adding a fistful of pecorino and black pepper.... and if you used the dish of CeramicaVietrese.it you'll se that they will be better if they are accompanied  with art and colors of Vietri ceramic!!!***\n\n[![MeTour.it](/tpl/default/assets/images/metour-banner-2025.jpg)](https://www.metour.it/it)\n\n##### Vietri Ceramic\n\n* [Vietri and the ceramic](/en/vietri-and-the-ceramic/)\n* [Our master potters](/en/our-master-potters/)\n* [The art of manufacturing ceramic](/en/the-art-of-manufacturing-ceramic/)\n* [Our design furniture](/en/our-design-furniture/)\n* [Photo Gallery](/en/photo-gallery/)\n* [Video-Gallery](/en/video-gallery/)\n* [Latest News](/en/news/)\n* [Recipes Claudia](/en/recipes-claudia/)\n* [Partner Area](/en/login-partner/)\n\n##### Our Shop\n\n* [Products Catalogue](/en/products/)\n* [Shipments](/en/shipments/)\n* [Payment methods](/en/payment-methods/)\n* [Warranty, defects and refunds](/en/warranty-defects-and-refunds/)\n* [Sure because we have MOCA certification](/en/sure-because-we-have-moca-certification/)\n* [FAQ Frequently Asked Questions](/en/faq-frequently-asked-questions/)\n\n##### Lists online\n\n* [Wedding list](/en/lists-online/wedding-list/)\n* [Gifts list](/en/lists-online/gifts-list/)\n* [Create a list](/en/login/?from=lc)\n* [Search a wedding list](/en/search-for-a-gift-list/?cat=wedding-list)\n* [Search for a gift list](/en/search-for-a-gift-list/?cat=gifts-list)\n\n##### Gift Card\n\n* [Give a Gift Card](/en/gift-card/)\n* [Ceramica Vietrese Card](/en/gift-card/ceramica-vietrese-card/)\n* [Mother's Day Card](/en/gift-card/mothers-day-card/)\n* [Happy Birthday Card](/en/gift-card/happy-birthday-card/)\n* [Merry Christmas Card](/en/gift-card/merry-christmas-card/)\n* [Happy Easter Card](/en/gift-card/happy-easter-card/)\n* [Wedding Gift Card](/en/gift-card/wedding-gift-card/)\n\n##### Customer Area\n\n* [User Login](/en/login/)\n* [Not registered?](/en/register-on-our-site/)\n\n##### Informations\n\n* [Terms of sale](/en/terms-of-sale/)\n* [Customer rights](/en/customer-rights/)\n* [Legal notes](/en/legal-notes/)\n* [Cookie policy](/en/cookie-policy/)\n* [Privacy policy](/en/privacy-policy/)\n\n##### Follow us on\n\n[![VK]()](http://vk.com/id344442374)\n\n##### Subscribe our newsletter\n\nYou will receive all updated news directly through your email account.\n\n##### Contacts\n\nCeramica Vietrese di  \n Di Palma Claudia  \n VAT 09299191214\n\n [Contact us](/en/contacts/)\n\n[![Italiano]( \"Italiano\")](/)  [![English]( \"English\")](/en)  [![Español]( \"Español\")](/es)  [![Deutsch]( \"Deutsch\")](/de)  [![Français]( \"Français\")](/fr)  [![Pусский]( \"Pусский\")](/ru)  [![中国]( \"中国\")](/c2)\n\n© 2026 Di Palma Claudia   2015 website by [dynform.it](http://www.dynform.it/)       \n& [studio150.it](http://www.studio150.it/)", "images": []}], "failed_results": [], "response_time": 0.01, "request_id": "548708e5-bc68-4aa1-b9ce-1a3f5d598f70"}

In [16]:
# DISPLAY AI MESSAGE
from IPython.display import Markdown, display

display(Markdown(response["messages"][-1].content))

SOURCE: https://tuduu.it/en/recipes/tuduu/pasta-alla-carbonara
![Pasta alla carbonara](https://tuduu-prd-assets-fde-ghdcd5e6baagctam.z01.azurefd.net/recipesimages/recipe-17278.jpg?v=2)

# Pasta alla carbonara

@tuduu

[Category: First courses](/en/recipes/categories/first-courses)

Pasta alla carbonara is a classic dish of Roman cuisine. It is a simple but irresistible first course, made with few ingredients but with an intense flavor. The creamy sauce is made with eggs, pancetta, Pecorino Romano and black pepper. Carbonara is a true comfort food, perfect to enjoy during cold winter evenings.

Difficulty: Easy

Cooking time: minCooking: min

Preparation time: 30 minPreparation: 30 min

Country: Italia


tuduu@tuduu

## Ingredients

No. Servings

* Spaghetti350350
* ![Guanciale](https://static.spaghettiemandolino.it/cdn-cgi/image/format=auto,fit=contain,width=500,height=500/img_prodotti/big/5731-guanciale-dolce-mendelspeck.png?1729593433)

  Guanciale120120
* ![Egg yolks](https://static.spaghettiemandolino.it/cdn-cgi/image/format=auto,fit=contain,width=500,height=500/img_prodotti/big/4912-zabaione-220g.png?1692365817)

  Egg yolksq.b.
* ![Pecorino Romano cheese](https://cdn.shopify.com/s/files/1/0025/0141/0873/products/Costanzo_Dispensa_0091_IMG_3041_risultato_jpg.jpg?v=1591617223)

  Pecorino romano cheese44
* ![Pepper](https://static.spaghettiemandolino.it/cdn-cgi/image/format=auto,fit=contain,width=500,height=500/img_prodotti/big/5004-aglio-olio-peperoncino-bustina-da-20g.png?1750088685)

  Pepperq.b.
* Saltq.b.

## Purchasable products

* ![Sweet Guanciale 250g - Mendelspeck](https://static.spaghettiemandolino.it/cdn-cgi/image/format=auto,fit=contain,width=500,height=500/img_prodotti/big/5731-guanciale-dolce-mendelspeck.png?1729593433)

  ### Sweet Guanciale 250g - Mendelspeck

  1 product1 product

  £ 5.48

* ![Zabaglione 220g](https://static.spaghettiemandolino.it/cdn-cgi/image/format=auto,fit=contain,width=500,height=500/img_prodotti/big/4912-zabaione-220g.png?1692365817)

  ### Zabaglione 220g

  1 product1 product

  £ 7.31

* ![Roman sheep cheese dop about 250 g - bronzetto](https://cdn.shopify.com/s/files/1/0025/0141/0873/products/Costanzo_Dispensa_0091_IMG_3041_risultato_jpg.jpg?v=1591617223)

  ### Roman sheep cheese dop about 250 g - bronzetto

  1 product1 product

  £ 7.40

* ![Garlic oil chili packet 20g](https://static.spaghettiemandolino.it/cdn-cgi/image/format=auto,fit=contain,width=500,height=500/img_prodotti/big/5004-aglio-olio-peperoncino-bustina-da-20g.png?1750088685)

  ### Garlic oil chili packet 20g

  1 product1 product

  £ 1.04

* ![Gray non-stick pan 1965 Vintage Quarzo Nero](https://static.spaghettiemandolino.it/cdn-cgi/image/format=auto,fit=contain,width=500,height=500/img_prodotti/big/2639-padella-antiaderente-grigia-1965-vintage-quarzo-nero.jpeg?1692363115)

  ### Gray non-stick pan 1965 Vintage Quarzo Nero

  £ 43.33

## Preparation

1. STEP 1 OF 3

   Dice the guanciale and brown it in a pan until it becomes crispy. Set aside some of its fat.
2. STEP 2 OF 3

   Beat the egg yolks with the Pecorino and pepper. Add the guanciale fat. Drain the pasta directly into the resulting creamy sauce and mix off the heat.
3. STEP 3 OF 3

   If necessary, add a ladle of the pasta cooking water for a creamier result. Serve with another grind of pepper and, if you like, more Pecorino Romano.

## Suggestions

* Pan
* Bowl

## General Information

### Origin

Italia, Lazio

## Analysis

## Macronutrients(100 gr)

## FAQs

|  |  |
| --- | --- |
| Energy (kcal) | 425.93 |
| Carbohydrates (g) | 48.82 |
| of which Sugars (g) | 2.2 |
| Fat (g) | 19.87 |
| of which Saturates (g) | 1.81 |
| Protein (g) | 12.55 |
| Fiber (g) | 1.82 |
| Sale (g) | 0.19 |

* Proteins

  12.55g · 12%
* Carbohydrates

  48.82g · 46%
* Fats

  19.87g · 42%

##### Attention

The data represented here, limited to certain specificities, are the result of an analysis carried out using Tuduu's proprietary algorithms. As such, they may contain errors and/or inaccuracies, therefore users are always requested to verify their correctness. If anomalies are detected, please contact us at[info@tuduu.it](mailto:info@tuduu.it)

Lifestyles

Intolerances and allergies


SOURCE: https://www.ceramicavietrese.it/en/recipes-claudia/a-typical-italian-food--pasta-alla-carbonara-a-typical-recipe-of-rome-tradition-and-italian-tradition/

![VIETRI CERAMICS - the excellence artisan pottery made in Italy](/files/admin/immagini/pasta-alla-carbonara-ricetta-tipica-in-ceramica-di-vietri-ceramica-vietrese4-l-20200910230002.jpg)
"Pork cheek"

## Recipes Claudia

[Back to the list](/en/recipes-claudia/)

# A typical Italian Food - Pasta alla "Carbonara" a typical recipe of Rome tradition and Italian tradition

A typical recipe of Italian traditions , especially Roman traditions . Easy to do, but that once tasted you can hardly forget . Tradition would like to use only seasoned and grated Pecorino Romano, but if you do not like the taste too strong , you can replace a portion of cheese with Original Parmigiano Reggiano (or original Grana Padano ) that definitely has a taste and a flavor less "strong " .

Pork cheek

Recipe for 4 people

***Ingredients:***

- 350 gr. of spaghetti of Gragnano

- 3 eggs + 1 egg yolk

- 160gr of cheek lard

- Roman pecorino

- Roman parmisan

- salt and pepper q.b.

***Preparazione:***

Put a pot with abundant water on the fire and when it is going to boile, you'll salt. Use less salt as usual because pasta have already a tasty condiment with cheek lard, parmesan and pecorino.

While you cut the cheek lard in cubes or strips, put in a pot on fire until the fatty become transparent and lightly crounch, then put off from the fire and let it lightly cold.

Break the eggs in a bowl and mix them. Add the pecorino, parmesan and the pepper (as you like) and mix to obtain an homogeneus mixture. Add the cheek lard.

Put the pasta to cook.

When pasta will be ready, dry it and put in the bowl and mix with the mixture and amalgamate them.

***Serve Spaghetti alla Carbonara immediately, adding a fistful of pecorino and black pepper.... and if you used the dish of CeramicaVietrese.it you'll se that they will be better if they are accompanied with art and colors of Vietri ceramic!!!***

##### Vietri Ceramic

* [Vietri and the ceramic](/en/vietri-and-the-ceramic/)
* [Our master potters](/en/our-master-potters/)
* [The art of manufacturing ceramic](/en/the-art-of-manufacturing-ceramic/)
* [Our design furniture](/en/our-design-furniture/)
* [Photo Gallery](/en/photo-gallery/)
* [Video-Gallery](/en/video-gallery/)
* [Latest News](/en/news/)
* [Recipes Claudia](/en/recipes-claudia/)
* [Partner Area](/en/login-partner/)

##### Our Shop

* [Products Catalogue](/en/products/)
* [Shipments](/en/shipments/)
* [Payment methods](/en/payment-methods/)
* [Warranty, defects and refunds](/en/warranty-defects-and-refunds/)
* [Sure because we have MOCA certification](/en/sure-because-we-have-moca-certification/)
* [FAQ Frequently Asked Questions](/en/faq-frequently-asked-questions/)

##### Lists online

* [Wedding list](/en/lists-online/wedding-list/)
* [Gifts list](/en/lists-online/gifts-list/)
* [Create a list](/en/login/?from=lc)
* [Search a wedding list](/en/search-for-a-gift-list/?cat=wedding-list)
* [Search for a gift list](/en/search-for-a-gift-list/?cat=gifts-list)

##### Gift Card

* [Give a Gift Card](/en/gift-card/)
* [Ceramica Vietrese Card](/en/gift-card/ceramica-vietrese-card/)
* [Mother's Day Card](/en/gift-card/mothers-day-card/)
* [Happy Birthday Card](/en/gift-card/happy-birthday-card/)
* [Merry Christmas Card](/en/gift-card/merry-christmas-card/)
* [Happy Easter Card](/en/gift-card/happy-easter-card/)
* [Wedding Gift Card](/en/gift-card/wedding-gift-card/)

##### Customer Area

* [User Login](/en/login/)
* [Not registered?](/en/register-on-our-site/)

##### Informations

* [Terms of sale](/en/terms-of-sale/)
* [Customer rights](/en/customer-rights/)
* [Legal notes](/en/legal-notes/)
* [Cookie policy](/en/cookie-policy/)
* [Privacy policy](/en/privacy-policy/)

##### Follow us on

##### Subscribe our newsletter

You will receive all updated news directly through your email account.

##### Contacts

Ceramica Vietrese di  
Di Palma Claudia  
VAT 09299191214

 [Contact us](/en/contacts/)

# SIMULATION OF A CONVERSATION

In [42]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Quanto pepe devo macinare per fare una carbonara perfetta?"}]},
    {"configurable": {"thread_id": "1"}},
)

In [44]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Quanto marsala è richiesto?"}]},
    {"configurable": {"thread_id": "1"}},
)

In [45]:
result

{'messages': [HumanMessage(content='text: "Pasta alla carbonara", params: {"max_results": 5, "topic": "general", "include_answer": false, "include_raw_content": false, "include_images": false, "include_image_descriptions": false, "search_depth": "basic", "time_range": "", "include_domains": ["*.it"], "exclude_domains": ["*.com", "wikipedia.org"], "country": "italy"}', additional_kwargs={}, response_metadata={}, id='908762fb-6994-4d1c-86e9-9d6155cf5cf9'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5:9b', 'created_at': '2026-04-07T11:03:52.2319872Z', 'done': True, 'done_reason': 'stop', 'total_duration': 137720816000, 'load_duration': 9443912600, 'prompt_eval_count': 1135, 'prompt_eval_duration': 44869451400, 'eval_count': 201, 'eval_duration': 83120056700, 'model_name': 'qwen3.5:9b', 'model_provider': 'ollama'}, id='lc_run--019d679a-d4de-73d2-8823-85b0be65fa72-0', tool_calls=[{'name': 'tavily_search', 'args': {'include_domains': ['*.it'], 'country':